# XFND-CLIP: model server for the website

This notebook runs your trained XFND-CLIP model on Colab's GPU and connects it to the website.

**Before starting:** `Runtime → Change runtime type → T4 GPU → Save`.

Run the boxes **from top to bottom**. Wait for the green tick ✅ before the next one.

| Box | What it does | First time | Demo day |
|---|---|---|---|
| 1 | Connects Google Drive and loads the project code | 2 min (upload `XFND-CLIP.zip`) | 1 min |
| 2 | Gets the trained model | 30–40 min (upload `archive.zip`, trains once, saves to Drive) | 1 min (loads from Drive) |
| 3 | Prepares the server | seconds | seconds |
| 4 | **Starts the model and prints the link for the website** | 2–3 min | 2–3 min |
| 5 | Stops the server | | |

Your files are kept in **Google Drive → `XFND`**, so you upload them only once.

## Box 1: connect Google Drive and load the project code
The first time, a **Choose Files** button appears: pick `XFND-CLIP.zip`.

In [ ]:
# ===== Box 1: connect Google Drive and load the project code =====
import os, glob, shutil, zipfile
from google.colab import drive, files

drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/XFND'
os.makedirs(DRIVE, exist_ok=True)

code_zip = os.path.join(DRIVE, 'XFND-CLIP.zip')
if not os.path.exists(code_zip):
    print('👉 Choose XFND-CLIP.zip (it is saved to Google Drive/XFND, so this happens only once)')
    uploaded = files.upload()
    shutil.move(next(iter(uploaded)), code_zip)

for d in ('/content/XFND-CLIP', '/content/tmp'):
    shutil.rmtree(d, ignore_errors=True)
zipfile.ZipFile(code_zip).extractall('/content/tmp')
shutil.move(os.path.dirname(glob.glob('/content/tmp/**/config.py', recursive=True)[0]), '/content/XFND-CLIP')

# Safety: load the saved model as data only (it can never run code).
import re
p = '/content/XFND-CLIP/explain.py'
s = open(p).read()
if 'weights_only' in s:
    print('Model loading is already safe (weights_only=True).')
else:
    s2 = re.sub(r'map_location=C\.DEVICE\)', 'map_location=C.DEVICE, weights_only=True)', s, count=1)
    open(p, 'w').write(s2)
    print('Model loading made safe (weights_only=True).' if s2 != s else '⚠️ Could not apply the safety patch.')

print('✅ Code ready - go to Box 2')

## Box 2: get the trained model
**First time:** pick `archive.zip` when asked. It trains once (30–40 min) and saves the model to Google Drive.

**Demo day:** loads the saved model in about a minute.

In [ ]:
# ===== Box 2: get the trained model =====
import os, glob, shutil, subprocess, zipfile
from google.colab import files
os.chdir('/content/XFND-CLIP')
subprocess.run('pip install -q -r requirements.txt', shell=True, check=True)
os.makedirs('checkpoints', exist_ok=True)
ck_drive = os.path.join(DRIVE, 'xfnd_clip.pt')

if os.path.exists(ck_drive):
    shutil.copy(ck_drive, 'checkpoints/xfnd_clip.pt')
    print('✅ Trained model loaded from Google Drive - go to Box 3')
else:
    print('No saved model yet: training once (30-40 min). Keep this tab open.')
    data_zip = os.path.join(DRIVE, 'archive.zip')
    if not os.path.exists(data_zip):
        print('👉 Choose archive.zip (the IFND dataset; saved to Google Drive/XFND)')
        uploaded = files.upload()
        shutil.move(next(iter(uploaded)), data_zip)
    shutil.rmtree('/content/ds', ignore_errors=True)
    zipfile.ZipFile(data_zip).extractall('/content/ds')
    csvs = glob.glob('/content/ds/**/*.csv', recursive=True)
    pick = [c for c in csvs if 'ifnd' in os.path.basename(c).lower()] or sorted(csvs, key=os.path.getsize)[-1:]
    os.makedirs('data', exist_ok=True)
    shutil.copy(pick[0], 'data/IFND.csv')
    for cmd in ['python data_prep.py', 'python features.py', 'python train.py']:
        print('\n>>> RUNNING:', cmd, flush=True)
        if subprocess.run(cmd, shell=True).returncode != 0:
            raise SystemExit(f'❌ Stopped at: {cmd} - send a screenshot of the error')
    shutil.copy('checkpoints/xfnd_clip.pt', ck_drive)
    if os.path.exists('results/metrics.json'):
        shutil.copy('results/metrics.json', os.path.join(DRIVE, 'metrics.json'))
    print('\n✅ Model trained and saved to Google Drive/XFND - next time Box 2 takes 1 minute. Go to Box 3')

## Box 3: prepare the server
Just run it. It writes `server.py`, the small program that answers the website.

In [ ]:
%%writefile /content/XFND-CLIP/server.py
"""XFND-CLIP model server: lets the website talk to the model running in Colab.

It wraps the project's own `explain.Explainer` (unchanged model code in ../model)
and returns everything the website shows as JSON:

  POST /api/analyse   form fields: text (the claim), image (optional file)
  GET  /api/health    is the model loaded? which GPU?

Every request must carry the access key the notebook prints (header
`X-Access-Key`), so only people with the connection code can use your GPU.

In Colab the notebook starts it with:
    XFND_ACCESS_KEY=... uvicorn server:app --port 8000
and exposes it with a temporary Cloudflare link.
"""

from __future__ import annotations

import base64
import io
import os
import secrets
import threading
import time
from typing import Callable

import numpy as np
from fastapi import FastAPI, File, Form, Header, HTTPException, Request, UploadFile
from fastapi.exceptions import RequestValidationError
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from PIL import Image, ImageOps, UnidentifiedImageError

API_VERSION = "1"
MAX_TEXT_CHARS = 1000
MIN_WORDS = 3
MAX_WORDS = 60   # BERT reads 64 tokens; words beyond that would get no importance
MAX_IMAGE_BYTES = 10 * 1024 * 1024
MAX_PIXELS = 24_000_000
PREVIEW_SIDE = 768
ALLOWED_FORMATS = {"JPEG", "PNG", "WEBP", "BMP", "GIF"}

Image.MAX_IMAGE_PIXELS = MAX_PIXELS


# ------------------------------------------------------------------ helpers
def load_upload(data: bytes) -> Image.Image:
    """Decode an uploaded image safely (format allow-list, pixel limit)."""
    try:
        img = Image.open(io.BytesIO(data))
        if (img.format or "").upper() not in ALLOWED_FORMATS:
            raise HTTPException(422, "Use a JPG, PNG, WEBP, BMP or GIF image.")
        # Check the size from the header BEFORE decoding (Pillow alone only warns
        # between 1x and 2x its limit).
        if img.width * img.height > MAX_PIXELS:
            raise HTTPException(422, "That image is too large.")
        img.load()
    except Image.DecompressionBombError:
        raise HTTPException(422, "That image is too large.") from None
    except (UnidentifiedImageError, OSError, SyntaxError):
        raise HTTPException(422, "This file couldn't be opened as an image.") from None
    img = ImageOps.exif_transpose(img).convert("RGB")
    if max(img.size) > 1600:  # the model works on 224 px; keep work and memory small
        img.thumbnail((1600, 1600), Image.Resampling.LANCZOS)
    return img


def data_url(img: Image.Image, fmt: str) -> str:
    buf = io.BytesIO()
    if fmt == "JPEG":
        img.save(buf, "JPEG", quality=85)
        mime = "image/jpeg"
    else:
        img.save(buf, "PNG", optimize=True)
        mime = "image/png"
    return f"data:{mime};base64,{base64.b64encode(buf.getvalue()).decode()}"


def heat_overlay(cam: np.ndarray) -> Image.Image:
    """Grad-CAM values 0..1 -> transparent-to-hot RGBA image.
    Low values are see-through so the photo shows; high values go yellow -> red."""
    v = np.clip(cam.astype(np.float32), 0, 1)
    s = np.clip((v - 0.2) / 0.6, 0, 1)
    rgba = np.stack([np.full_like(s, 255), 230 - 190 * s, 60 * (1 - s), s * 210], axis=-1)
    return Image.fromarray(rgba.astype(np.uint8), "RGBA")


def shrink(img: Image.Image, side: int) -> Image.Image:
    out = img.copy()
    out.thumbnail((side, side), Image.Resampling.LANCZOS)
    return out


# ------------------------------------------------------------------ app
def create_app(explainer_factory: Callable[[], object], access_key: str,
               allowed_origins: list[str] | None = None, device_info: dict | None = None) -> FastAPI:
    """Build the API. `explainer_factory` returns an object with the same
    methods as model/explain.py's Explainer (tests pass a fake one)."""
    if not access_key or len(access_key) < 16:
        raise ValueError("Set a long random access key.")

    app = FastAPI(title="XFND-CLIP model server", docs_url=None, redoc_url=None, openapi_url=None)
    state = {"explainer": None, "error": None}
    gpu_lock = threading.Lock()  # one analysis at a time on the GPU

    def explainer():
        if state["explainer"] is None:
            with gpu_lock:
                if state["explainer"] is None and state["error"] is None:
                    try:
                        state["explainer"] = explainer_factory()
                    except Exception as exc:  # e.g. missing checkpoint, download failure
                        state["error"] = f"{type(exc).__name__}: {exc}"
        if state["explainer"] is None:
            raise HTTPException(500, f"The model failed to load. {state['error']}")
        return state["explainer"]

    def check_key(key: str | None) -> None:
        if not key or not secrets.compare_digest(key, access_key):
            raise HTTPException(401, "Wrong or missing access key. Copy the connection code from the notebook again.")

    @app.middleware("http")
    async def guard(request: Request, call_next):
        # Check the key before anything is read or parsed (CORS preflights carry no key).
        if request.url.path.startswith("/api/") and request.method != "OPTIONS":
            key = request.headers.get("x-access-key") or ""
            if not secrets.compare_digest(key, access_key):
                return JSONResponse({"detail": "Wrong or missing access key. Copy the connection code from the notebook again."}, 401)
        if request.method == "POST":
            length = request.headers.get("content-length")
            if length is None:
                return JSONResponse({"detail": "Content-Length header is required."}, 411)
            if not length.isdigit() or int(length) > MAX_IMAGE_BYTES + 64 * 1024:
                return JSONResponse({"detail": "That upload is too large (10 MB limit)."}, 413)
        return await call_next(request)

    @app.get("/api/health")
    def health(x_access_key: str | None = Header(default=None)):
        check_key(x_access_key)
        ex = explainer()
        return {"status": "ok", "api_version": API_VERSION, "model": "xfnd_clip",
                "clip_sim_mean": round(float(ex.sim_mean), 4), **(device_info or {})}

    @app.post("/api/analyse")
    def analyse(text: str = Form(...), image: UploadFile | None = File(default=None),
                x_access_key: str | None = Header(default=None)):
        check_key(x_access_key)
        text = " ".join(text.split())
        if len(text) > MAX_TEXT_CHARS:
            raise HTTPException(422, f"Keep the claim under {MAX_TEXT_CHARS} characters.")
        if len(text.split()) < MIN_WORDS:
            raise HTTPException(422, f"Type the full claim (at least {MIN_WORDS} words).")
        if len(text.split()) > MAX_WORDS:
            raise HTTPException(422, f"Keep the claim to {MAX_WORDS} words or fewer (the model reads a headline, not an article).")

        img = None
        if image is not None and image.filename:
            raw = image.file.read(MAX_IMAGE_BYTES + 1)
            if len(raw) > MAX_IMAGE_BYTES:
                raise HTTPException(413, "That image is over 10 MB.")
            if raw:
                img = load_upload(raw)

        ex = explainer()
        t0 = time.perf_counter()
        try:
            with gpu_lock:
                pred = ex.predict(text, img)
                words = ex.word_importance(text, img)
                cam = ex.gradcam(text, img) if img is not None else None
        except Exception as exc:  # e.g. GPU out of memory; answer properly so the site can show it
            raise HTTPException(500, f"The model failed on this post ({type(exc).__name__}). Try again or use a smaller image.") from None
        elapsed = int((time.perf_counter() - t0) * 1000)

        image_out = None
        if img is not None:
            view = shrink(img, PREVIEW_SIDE)
            heat = Image.fromarray((np.clip(cam, 0, 1) * 255).astype(np.uint8)).resize(view.size, Image.Resampling.BILINEAR)
            image_out = {
                "preview": data_url(view, "JPEG"),
                "heatmap": data_url(heat_overlay(np.asarray(heat, np.float32) / 255), "PNG"),
                "width": img.width, "height": img.height,
            }
        sim = pred.get("clip_sim")
        sim_mean = float(ex.sim_mean)
        return {
            "api_version": API_VERSION,
            "claim": text,
            "label": pred["label"],
            "prob_fake": round(float(pred["prob_fake"]), 4),
            "modality": {"text": round(float(pred["att_text"]), 4),
                         "image": round(float(pred["att_image"]), 4),
                         "text_image": round(float(pred["att_mm"]), 4)},
            "clip_sim": None if sim is None else round(float(sim), 4),
            "clip_sim_mean": round(sim_mean, 4),
            "clip_match": None if sim is None else ("match" if sim >= sim_mean else "weak"),
            "words": [{"word": w, "weight": round(float(v), 4)} for w, v in words],
            "image": image_out,
            "elapsed_ms": elapsed,
        }

    @app.exception_handler(HTTPException)
    async def http_error(request: Request, exc: HTTPException):
        return JSONResponse({"detail": exc.detail}, exc.status_code)

    @app.exception_handler(RequestValidationError)
    async def bad_request(request: Request, exc: RequestValidationError):
        return JSONResponse({"detail": "Send the claim as 'text' (and optionally an 'image' file)."}, 422)

    # Added last so it wraps everything, including the size-limit replies above:
    # otherwise the browser would see a CORS failure instead of the message.
    app.add_middleware(
        CORSMiddleware,
        allow_origins=allowed_origins or ["*"],
        allow_methods=["GET", "POST"],
        allow_headers=["X-Access-Key"],
        max_age=600,
    )
    return app


def _colab_app() -> FastAPI:
    """The real app used in Colab (imports torch and the model code lazily)."""
    import torch

    from explain import Explainer

    info = {"device": "cuda" if torch.cuda.is_available() else "cpu",
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
    origins = [o.strip() for o in os.environ.get("XFND_ALLOWED_ORIGINS", "*").split(",") if o.strip()]
    return create_app(Explainer, os.environ.get("XFND_ACCESS_KEY", ""), origins, info)


if os.environ.get("XFND_ACCESS_KEY"):
    app = _colab_app()

## Box 4: start the model and get the link
Wait for the green box, click **open the website**, then press **Connect**.

First time only: after deploying the website, put its address in `SITE_URL` below.

In [ ]:
# ===== Box 4: start the model and get the link for the website =====
# Your website address (the Render link). Change it once after deploying the site.
SITE_URL = 'https://xfnd-clip.onrender.com'

import os, re, secrets, subprocess, time, urllib.error, urllib.parse, urllib.request
from IPython.display import HTML, display
os.chdir('/content/XFND-CLIP')
subprocess.run('pip install -q fastapi "uvicorn[standard]" python-multipart', shell=True, check=True)
def stop_old():
    # No shell here: "pkill -f" inside a shell would also match (and kill) that shell.
    subprocess.run(['pkill', '-f', 'uvicorn server:app'])
    subprocess.run(['pkill', '-f', 'cloudflared tunnel'])
    import socket
    for _ in range(20):                      # wait until port 8000 is free again
        with socket.socket() as s:
            if s.connect_ex(('127.0.0.1', 8000)) != 0:
                return
        time.sleep(0.5)
stop_old()

KEY = secrets.token_urlsafe(24)   # new secret every run: only people with the link can use your GPU
# Only your website may call the server from a browser.
ORIGINS = ','.join([SITE_URL.rstrip('/'), 'http://127.0.0.1:8080', 'http://localhost:8080'])
server = subprocess.Popen(['python', '-m', 'uvicorn', 'server:app', '--host', '127.0.0.1', '--port', '8000'],
                          env=dict(os.environ, XFND_ACCESS_KEY=KEY, XFND_ALLOWED_ORIGINS=ORIGINS),
                          stdout=open('server.log', 'w'), stderr=subprocess.STDOUT)

print('Loading BERT, ResNet-50, CLIP and your model (about 1-2 minutes)...', flush=True)
deadline = time.time() + 600
while True:
    if server.poll() is not None:
        print(open('server.log').read()[-3000:])
        raise SystemExit('❌ The server stopped. Send a screenshot of the text above.')
    try:
        req = urllib.request.Request('http://127.0.0.1:8000/api/health', headers={'X-Access-Key': KEY})
        if urllib.request.urlopen(req, timeout=300).status == 200:
            break
    except urllib.error.HTTPError as err:
        if err.code == 500:                  # the model could not load: show why and stop
            print(err.read().decode(errors='replace'))
            print(open('server.log').read()[-3000:])
            raise SystemExit('❌ The model failed to load (see above). Did Box 2 finish with a green tick?')
        time.sleep(3)
    except Exception:
        if time.time() > deadline:
            raise SystemExit('❌ The model took too long to load. Run Box 4 again.')
        time.sleep(3)
print('✅ Model loaded')

# Cloudflare's tunnel program, a fixed version checked against its published SHA-256.
CF_URL = 'https://github.com/cloudflare/cloudflared/releases/download/2026.11.0/cloudflared-linux-amd64'
CF_SHA256 = '51f7c68b57ed7269164075d7e789efce5b59355982f06392deeb037d982a51fd'
import hashlib
if not os.path.exists('cloudflared') or hashlib.sha256(open('cloudflared', 'rb').read()).hexdigest() != CF_SHA256:
    urllib.request.urlretrieve(CF_URL, 'cloudflared')
    if hashlib.sha256(open('cloudflared', 'rb').read()).hexdigest() != CF_SHA256:
        os.remove('cloudflared')
        raise SystemExit('❌ The tunnel program failed its checksum. Run Box 4 again.')
    os.chmod('cloudflared', 0o755)
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
                          stdout=open('tunnel.log', 'w'), stderr=subprocess.STDOUT)
url = None
for _ in range(90):
    m = re.search(r'https://(?!api\.)[a-z0-9-]+\.trycloudflare\.com', open('tunnel.log').read())
    if m:
        url = m.group(0)
        break
    time.sleep(1)
if not url:
    raise SystemExit('❌ Could not create the public link. Run Box 4 again.')

CODE = f'{url}#key={KEY}'
LINK = f"{SITE_URL.rstrip('/')}/#connect={urllib.parse.quote(CODE, safe='')}"
display(HTML(f'''
<div style="font:15px/1.5 sans-serif;padding:16px;border:2px solid #0f9d58;border-radius:10px">
  <b style="font-size:18px">✅ The model is online</b><br><br>
  <b>Easiest:</b> <a href="{LINK}" target="_blank" style="font-size:16px">open the website</a> and press <b>Connect</b><br><br>
  <b>Or</b> copy this connection code into the website's <i>Connect model</i> box:<br>
  <code style="user-select:all;word-break:break-all">{CODE}</code><br><br>
  Works only from <b>{SITE_URL}</b>. Keep this Colab tab open during the demo. Stop with Box 5.<br>
  <small>Before sharing this notebook with anyone: <i>Edit → Clear all outputs</i> (this box contains the secret key).</small>
</div>'''))

## Box 5: stop the server (after the demo)

In [ ]:
# ===== Box 5: stop the server =====
import subprocess
subprocess.run(['pkill', '-f', 'uvicorn server:app'])
subprocess.run(['pkill', '-f', 'cloudflared tunnel'])
print('Stopped. The website now shows the model as offline.')